# Event-Episode v2 — Colab Runner

End-to-end pipeline for the v2 event-episode experiment.  Runs:

1. **Tabular HPO** (Phase 4) — LR/RF/XGB sweep over (T, L, W).
2. **DL HPO + multi-seed** (Phase 5) — GRU/LSTM/RNN/CNN.
3. **Sensor ablation** (Phase 6).
4. **Leakage probe** (Phase 7) — gate before final test.
5. **Final test evaluation** (Phase 8).

All outputs are written under `outputs/v2/tables/`.  This notebook does **not** modify any v1 file.

## 1. Clone / pull repo, install deps, mount Drive

In [8]:
!nvidia-smi


Thu May 14 21:21:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
from pathlib import Path

REPO_URL = "https://github.com/bahaa1515/EECE-693-project.git"
REPO_DIR = Path("/content/EECE-693-project")
BRANCH = "codex/tarek-event-episode-tuning"

if REPO_DIR.exists():
    %cd /content/EECE-693-project
    !git fetch --all --quiet
    !git checkout {BRANCH}
    !git pull --ff-only
else:
    %cd /content
    !git clone {REPO_URL} EECE-693-project
    %cd /content/EECE-693-project
    !git checkout {BRANCH}

/content/EECE-693-project
Already on 'codex/tarek-event-episode-tuning'
Your branch is up to date with 'origin/codex/tarek-event-episode-tuning'.
Already up to date.


In [10]:
!pip install -q -r requirements-colab.txt

In [11]:
import sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Python: 3.12.13
Torch: 2.4.0+cu121
CUDA: True Tesla T4


In [12]:
# Mount Drive and point at the AAMOS dataset / output folder.
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Drive mount skipped:", exc)

data_dir = Path(os.environ.get("AAMOS_DATA_DIR", "/content/drive/MyDrive/AAMOS/dataset"))
out_dir  = Path(os.environ.get("AAMOS_OUTPUT_DIR", "/content/drive/MyDrive/AAMOS/outputs"))
if data_dir.exists():
    os.environ["AAMOS_DATA_DIR"] = str(data_dir)
    os.environ["AAMOS_OUTPUT_DIR"] = str(out_dir)
    print("AAMOS_DATA_DIR =", os.environ["AAMOS_DATA_DIR"])
    print("AAMOS_OUTPUT_DIR =", os.environ["AAMOS_OUTPUT_DIR"])
else:
    print("Set AAMOS_DATA_DIR to the Drive folder containing the AAMOS CSVs.")

Drive mount skipped: Failed to issue request POST https://colab.research.google.com/tun/m/credentials-propagation/gpu-t4-s-kkb-usw1b1-1iy8tcdmcbhpn?authtype=dfs_ephemeral&version=2&dryrun=false&propagate=true&record=false&authuser=0: Bad Request
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 400 (Bad Request)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/e

## 2. Configure the experiment

Set the (T, L, W) grid here.  Smaller grids run faster on Colab CPUs;
the defaults match the plan.

In [13]:
THRESHOLDS = "2,3,4"
LENGTHS    = "3,7,14"      # L=28 omitted for DL (plan); tabular tolerates it
WASHOUTS   = "0,7,14"
SEED       = 42
DL_EPOCHS  = 30
DL_BATCH   = 32
N_SHUFFLES = 5
MULTI_SEEDS = "42,43,44,45,46"

## 3. One-command protocol runner

Use this cell when you want Colab to run the complete v2 protocol from the shared launcher. The phase-by-phase cells below are kept for debugging or partial reruns.

In [14]:
!python scripts/v2/run_metric_protocol.py full-v2 --dry-run

[v2-protocol] profile=full-v2 quick=False dry_run=True
[v2-protocol] cwd=/content/EECE-693-project

[1/7] /usr/bin/python3 -m compileall src/event_v2 scripts/v2

[2/7] /usr/bin/python3 -m scripts.v2.tune_tabular_v2 --thresholds 2,3,4 --lengths 3,7,14 --washouts 0,7,14 --algos lr,rf,xgb --seed 42

[3/7] /usr/bin/python3 -m scripts.v2.tune_dl_v2 --thresholds 2,3,4 --lengths 3,7,14 --washouts 0,7,14 --archs gru,lstm,rnn,cnn --seed 42 --multi-seeds 42,43,44,45,46 --epochs 30 --batch-size 32 --patience 5 --learning-rate 0.0005

[4/7] /usr/bin/python3 -m scripts.v2.leakage_probe_v2 --n-shuffles 5 --seed 42

[5/7] /usr/bin/python3 -m scripts.v2.sensor_ablation_v2 --seed 42

[6/7] /usr/bin/python3 -m scripts.v2.final_test_eval_v2 --seed 42 --dl-batch-size 32 --dl-patience 5 --dl-learning-rate 0.0005

[7/7] /usr/bin/python3 -m scripts.v2.analyze_v2 --n-boot 2000


In [15]:
# Heavy: starts tabular, DL, leakage, ablation, final evaluation, and analysis.
# Run this only after the dry-run command looks correct.
!python scripts/v2/run_metric_protocol.py full-v2

[v2-protocol] profile=full-v2 quick=False dry_run=False
[v2-protocol] cwd=/content/EECE-693-project

[1/7] /usr/bin/python3 -m compileall src/event_v2 scripts/v2
Listing 'src/event_v2'...
Compiling 'src/event_v2/__init__.py'...
Compiling 'src/event_v2/deep_learning_v2.py'...
Compiling 'src/event_v2/features_v2.py'...
Compiling 'src/event_v2/modeling_v2.py'...
Compiling 'src/event_v2/samples_v2.py'...
Compiling 'src/event_v2/split_v2.py'...
Listing 'scripts/v2'...
Compiling 'scripts/v2/__init__.py'...
Compiling 'scripts/v2/analyze_v2.py'...
Compiling 'scripts/v2/final_test_eval_v2.py'...
Compiling 'scripts/v2/leakage_probe_v2.py'...
Compiling 'scripts/v2/run_metric_protocol.py'...
Compiling 'scripts/v2/sensor_ablation_v2.py'...
Compiling 'scripts/v2/tune_dl_v2.py'...
Compiling 'scripts/v2/tune_tabular_v2.py'...

[2/7] /usr/bin/python3 -m scripts.v2.tune_tabular_v2 --thresholds 2,3,4 --lengths 3,7,14 --washouts 0,7,14 --algos lr,rf,xgb --seed 42
[v2-tune-tabular] outputs -> /content/EECE

## 4. Phase 4 — Tabular HPO

In [ ]:
!python -m scripts.v2.tune_tabular_v2 \
    --thresholds {THRESHOLDS} \
    --lengths {LENGTHS} \
    --washouts {WASHOUTS} \
    --seed {SEED}

[v2-tune-tabular] outputs -> /content/EECE-693-project/outputs/v2/tables
  thresholds=[2, 3, 4]  lengths=[3, 7, 14]  washouts=[0, 7, 14]
  algos=['lr', 'rf', 'xgb']  seed=42  sensor_tag=all

[1/4] Building event labels...

[2/4] Building v2 sample indexes...

[3/4] Building v2 feature tables...


In [ ]:
import pandas as pd
best = pd.read_csv("outputs/v2/tables/tune_tabular_v2_best.csv")
best.sort_values("val_pr_auc", ascending=False).head(20)

## 5. Phase 5 — Deep-learning HPO + multi-seed

In [ ]:
!python -m scripts.v2.tune_dl_v2 \
    --thresholds {THRESHOLDS} \
    --lengths {LENGTHS} \
    --washouts {WASHOUTS} \
    --seed {SEED} \
    --multi-seeds {MULTI_SEEDS} \
    --epochs {DL_EPOCHS} \
    --batch-size {DL_BATCH}

In [ ]:
ms = pd.read_csv("outputs/v2/tables/tune_dl_v2_multiseed.csv")
ms

## 6. Phase 6 — Sensor ablation

In [ ]:
!python -m scripts.v2.sensor_ablation_v2

In [ ]:
abl = pd.read_csv("outputs/v2/tables/sensor_ablation_v2.csv")
abl.sort_values("val_pr_auc", ascending=False).head(30)

## 7. Phase 7 — Leakage probe (gate before Phase 8)

If any winner's shuffled-label ROC-AUC falls outside `[0.45, 0.55]`,
this prints a WARNING and you should investigate before trusting the
Phase-8 numbers.

In [ ]:
!python -m scripts.v2.leakage_probe_v2 --n-shuffles {N_SHUFFLES}

## 8. Phase 8 — Final test-set evaluation

In [ ]:
!python -m scripts.v2.final_test_eval_v2 \
    --dl-epochs {DL_EPOCHS} \
    --dl-batch-size {DL_BATCH}

In [ ]:
summary = pd.read_csv("outputs/v2/tables/final_test_v2_summary.csv")
summary

## 9. Phase 9 — Offline analysis (bootstrap CIs, v1-vs-v2, leakage, ablation)

Reads the predictions and summary CSVs from Phase 8 and computes:
- Bootstrap 95% CIs for test PR-AUC / ROC-AUC.
- v1-vs-v2 comparison on matching (T, L, W).
- Leakage-probe overall mean check (should be near 0.5).
- Sensor-ablation top-K per algo.

No training; runs in seconds.

In [ ]:
!python -m scripts.v2.analyze_v2 --n-boot 2000

In [ ]:
from pathlib import Path
tbl = Path("outputs/v2/tables")
for name in ["analysis_v2_tabular_ci.csv", "analysis_v2_dl_ci.csv", "analysis_v2_v1_vs_v2.csv", "analysis_v2_sensor_ablation_topk.csv"]:
    pth = tbl / name
    if pth.exists():
        print(f"\n--- {name} ---")
        display(pd.read_csv(pth))

## 10. (Optional) Copy v2 outputs back to Drive

Useful when Colab's local FS will be wiped at session end.

In [ ]:
import shutil
drive_target = Path("/content/drive/MyDrive/AAMOS/outputs/v2")
if drive_target.parent.exists():
    drive_target.mkdir(parents=True, exist_ok=True)
    shutil.copytree("outputs/v2", drive_target, dirs_exist_ok=True)
    print("copied -> ", drive_target)
else:
    print("Drive output folder not present; skipping copy.")